# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells: Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and analyzing the FAIR^2 dataset using the [`mlcroissant`](https://mlcroissant.ai/) library. You will learn how to:

- Load dataset schema and metadata
- Explore available record sets, fields, and IDs via their `@id`
- Extract and process records from specified record sets
- Perform exploratory data analysis and visualize dataset statistics

### Dataset Source
The dataset schema is provided as a Croissant-compliant JSON-LD at:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json` 

*Cite as: Kulka, M., Rodriguez Miera, S. & Marcet-Palacios, M. (2026). Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.*

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load schema, metadata, and records from the dataset via `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# Define the FAIR^2 Croissant JSON-LD schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset's schema and metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"Title: {metadata.name}")
print(f"Description: {metadata.description[:200]}...")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Record sets: {len(metadata.recordSet) if hasattr(metadata, 'recordSet') and metadata.recordSet else 0}")
print(f"Keywords: {getattr(metadata, 'keywords', 'N/A')}")

## 2. Data Overview
Let's examine the available record sets, the fields (columns) contained in them, and their Croissant `@id`s. This helps you understand the schema and refer to entities precisely.

In [ ]:
# List available record sets and their fields by @id
if not metadata.recordSet:
    print("No record sets found in the schema. Please inspect the schema directly.")
else:
    for rs in metadata.recordSet:
        rs_id = rs['@id']
        rs_name = rs.get('name', 'unnamed')
        print(f"RecordSet: @id={rs_id}\n  Name: {rs_name}\n  Fields and columns:")
        for field in rs.get('field', []):
            field_id = field['@id']
            label = field.get('name', field_id)
            dtype = field.get('dataType', 'unknown')
            col_ids = [col['@id'] for col in field.get('column', [])] if 'column' in field else []
            print(f"   - Field: @id={field_id}  |  Name: {label}  |  Type: {dtype}  |  Col @ids: {col_ids}")


> **Note:**
If no record sets print (empty), please inspect the schema directly as the provided copy has empty `recordSet` and you may need to consult the actual schema online.

Below, we attempt to print some examples from one available record set using the correct `@id`. 

In [ ]:
# --- Example: Preview records from a record set ---
# First, auto-discover available record sets
def get_first_record_set(metadata):
    if hasattr(metadata, 'recordSet') and metadata.recordSet:
        for rs in metadata.recordSet:
            if '@id' in rs:
                return rs['@id']
    return None

record_set_id = get_first_record_set(metadata)

if record_set_id:
    print(f"Previewing data from RecordSet @id={record_set_id}")
    for i, rec in enumerate(dataset.records(record_set=record_set_id)):
        print(rec)
        if i >= 2:
            break
else:
    print("No available record set found by @id.")

## 3. Data Extraction
Extract records from all available record sets. All entities are referenced via their `@id` per Croissant best practices. Records are loaded into pandas DataFrames for further analysis.

In [ ]:
# Collect all record set @ids
record_set_ids = []
if hasattr(metadata, 'recordSet') and metadata.recordSet:
    record_set_ids = [rs['@id'] for rs in metadata.recordSet if '@id' in rs]

print(f"Record set @ids: {record_set_ids}")

dataframes = dict()
for rs_id in record_set_ids:
    # This will only work if records exist in the recordSet
    df = pd.DataFrame(dataset.records(record_set=rs_id))
    dataframes[rs_id] = df
    print(f"Loaded DataFrame with shape {df.shape} for record set {rs_id}")

# Demo: show column names and a preview for the first record set
if record_set_ids:
    fid = record_set_ids[0]
    print(f"Available columns for {fid}: {dataframes[fid].columns.tolist() if not dataframes[fid].empty else 'None (empty DataFrame)'}")
    display(dataframes[fid].head())
else:
    print("No RecordSets loaded.")

## 4. Exploratory Data Analysis (EDA)
You can filter, transform, and analyze columns by referencing their Croissant `@id`. Demo below (edit to match field IDs in your schema):
- Filter numerics
- Normalize values
- Group by attribute

In [ ]:
# --- EXAMPLE: Filter and normalize a numeric column ---
import numpy as np

# Choose a record set and field/column for demonstration
# (Supply correct @ids for {numeric_field_id} and {group_field_id} after inspecting your schema)

if record_set_ids:
    rsid = record_set_ids[0]
    df = dataframes[rsid]

    # If DataFrame is not empty, pick a numeric column
    if not df.empty:
        # Guess the first numeric column
        numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
        if numeric_columns:
            numeric_field_id = numeric_columns[0]
            print(f"Filtering on numeric field: {numeric_field_id}")

            threshold = df[numeric_field_id].mean() if df[numeric_field_id].mean() > 0 else 10
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold:.2f} (showing up to 5):")
            display(filtered_df.head())

            # Normalize field
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records (showing up to 5):")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Attempt groupby
            # Guess a categorical column
            categ_columns = df.select_dtypes(include=['object', 'category']).columns.tolist()
            group_field_id = categ_columns[0] if categ_columns else None
            if group_field_id:
                grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
                print(f"Grouped data by {group_field_id} (showing up to 5):")
                display(grouped_df.head())
            else:
                print("No suitable categorical field for groupby found.")
        else:
            print("No numeric fields found in this record set.")
    else:
        print("The loaded DataFrame is empty; cannot demonstrate EDA.")
else:
    print("No record set available to analyze.")

## 5. Visualization
Visualize field distributions or relationships. Use `@id` column headers for reference. Edit this code as needed to match the chosen field IDs.

In [ ]:
# --- EXAMPLE: Histogram and boxplot of numeric field ---
import matplotlib.pyplot as plt
import seaborn as sns

if record_set_ids and not dataframes[record_set_ids[0]].empty:
    rsid = record_set_ids[0]
    df = dataframes[rsid]
    # Guess numeric field
    numeric_columns = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_columns:
        col = numeric_columns[0]
        plt.figure(figsize=(12,4))
        plt.subplot(1,2,1)
        sns.histplot(df[col].dropna(), kde=True)
        plt.title(f'Histogram for "{col}" (@id)')
        plt.xlabel(col)
        plt.ylabel('Frequency')
        plt.subplot(1,2,2)
        sns.boxplot(x=df[col].dropna())
        plt.title(f'Boxplot for "{col}" (@id)')
        plt.tight_layout()
        plt.show()
    else:
        print("No numeric columns available for visualization.")
else:
    print("No non-empty record sets for visualization.")

## 6. Conclusion
We explored the FAIR^2 mass spectrometry dataset via Croissant schema using `mlcroissant`. 
We programmatically identified record sets, referenced all fields by their `@id`, and performed basic analysis and visualizations. 

- For more advanced workflows, edit the field IDs in the notebook cells to match your analytical questions.
- For schema inspection or issues, view the Croissant JSON-LD directly: [https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json).

🚀 *Happy FAIR and reproducible data science!*